In [1]:
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# 社員ごとの売上データ
data = {
    "日付": ["2023-05-17", "2023-05-18", "2023-05-19", "2023-05-20", "2023-05-21"],
    "社員名": ["山田", "佐藤", "鈴木", "田中", "高橋"],
    "売上": [100, 200, 150, 300, 250],
    "部門": ["メーカー", "代理店", "メーカー", "商社", "代理店"]
}

df = pd.DataFrame(data)

# 全体の売上平均を算出
avg_sales = int(df["売上"].mean())
df["平均売上"] = avg_sales

# 業績ランクを判定する関数
def performance(sales, avg):
    if sales >= avg + 50:
        return "A"
    elif sales >= avg:
        return "B"
    else:
        return "C"

df["業績ランク"] = df.apply(lambda row: performance(row["売上"], row["平均売上"]), axis=1)

print(df.to_string(index=False))

# Excelファイルに書き出し
output_path = "業績.xlsx"
df.to_excel(output_path, index=False, sheet_name="業績データ")

# openpyxlで書式設定
wb = load_workbook(output_path)
ws = wb.active

# スタイル定義
header_font = Font(name="Arial", bold=True, color="FFFFFF", size=11)
header_fill = PatternFill("solid", start_color="2E75B6")
data_font = Font(name="Arial", size=10)
center_align = Alignment(horizontal="center", vertical="center")
left_align = Alignment(horizontal="left", vertical="center")

rank_colors = {"A": "C6EFCE", "B": "FFEB9C", "C": "FFC7CE"}
rank_fonts  = {"A": "276221", "B": "9C5700", "C": "9C0006"}

thin = Side(style="thin", color="BFBFBF")
border = Border(left=thin, right=thin, top=thin, bottom=thin)

# ヘッダー行の書式
for col in range(1, ws.max_column + 1):
    cell = ws.cell(row=1, column=col)
    cell.font = header_font
    cell.fill = header_fill
    cell.alignment = center_align
    cell.border = border

# データ行の書式
for row in range(2, ws.max_row + 1):
    rank = ws.cell(row=row, column=6).value  # 業績ランク列
    for col in range(1, ws.max_column + 1):
        cell = ws.cell(row=row, column=col)
        cell.font = data_font
        cell.border = border
        cell.alignment = center_align if col in [1, 3, 5, 6] else left_align
    # 業績ランク列のセル色
    if rank in rank_colors:
        rank_cell = ws.cell(row=row, column=6)
        rank_cell.fill = PatternFill("solid", start_color=rank_colors[rank])
        rank_cell.font = Font(name="Arial", size=10, bold=True, color=rank_fonts[rank])

# 列幅を設定
col_widths = [14, 10, 10, 12, 14, 12]
for i, width in enumerate(col_widths, 1):
    ws.column_dimensions[get_column_letter(i)].width = width

# 行の高さ
ws.row_dimensions[1].height = 22
for row in range(2, ws.max_row + 1):
    ws.row_dimensions[row].height = 18

wb.save(output_path)
print(f"\n平均売上: {avg_sales}")
print("完了: 業績.xlsx を作成しました")

        日付 社員名  売上   部門  平均売上 業績ランク
2023-05-17  山田 100 メーカー   200     C
2023-05-18  佐藤 200  代理店   200     B
2023-05-19  鈴木 150 メーカー   200     C
2023-05-20  田中 300   商社   200     A
2023-05-21  高橋 250  代理店   200     A

平均売上: 200
完了: 業績.xlsx を作成しました
